# QM9 `DataSource` & `Batch` Demo

Loads a subset of the QM9 validation split via `miniqm.data_source.DataSource`, inspects the padded `Batch` PyTree tensors, verifies zero-COM coordinate centering, and visualizes the featurized 2D molecular graphs.

In [ ]:
import pathlib
from typing import Final

import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw
import seaborn as sns

from miniqm.data import data_source, schema

DATA_DIR: Final = pathlib.Path("../data").resolve()
VALIDATION_PATH: Final = DATA_DIR / "validation.parquet"

source = data_source.DataSource(VALIDATION_PATH, max_molecules=256)
print(f"Loaded DataSource with {len(source)} molecules.")

In [ ]:
rng = np.random.default_rng(42)
sample_batch = next(source.iter_batches(batch_size=32, rng=rng))

print("Batch field shapes & dtypes:")
print(f"  atom_features : {sample_batch.atom_features.shape} ({sample_batch.atom_features.dtype})")
print(f"  bond_features : {sample_batch.bond_features.shape} ({sample_batch.bond_features.dtype})")
print(f"  atom_mask     : {sample_batch.atom_mask.shape} ({sample_batch.atom_mask.dtype})")
print(f"  positions     : {sample_batch.positions.shape} ({sample_batch.positions.dtype})")
print(f"  labels        : {sample_batch.labels.shape} ({sample_batch.labels.dtype})")

masked_com = (sample_batch.positions * sample_batch.atom_mask[..., None]).sum(axis=1)
print(f"\nMax absolute coordinate sum over valid atoms: {np.abs(masked_com).max():.2e} Å")

In [ ]:
molecules = schema.load_molecules(VALIDATION_PATH, max_molecules=8)
rdkit_mols = tuple(
    Chem.MolFromSmiles(mol.smiles_relaxed) or Chem.MolFromSmiles(mol.smiles) for mol in molecules
)
legends = tuple(f"idx={mol.index} (N={mol.num_atoms})\n{mol.smiles_relaxed}" for mol in molecules)

Draw.MolsToGridImage(rdkit_mols, molsPerRow=4, subImgSize=(240, 200), legends=legends)

In [ ]:
full_batch = next(source.iter_batches(batch_size=len(source)))
num_atoms_per_mol = full_batch.atom_mask.sum(axis=1)
# Column 7 in QM9Labels is the HOMO-LUMO gap (Hartree).
homo_lumo_gap = full_batch.labels[:, 7]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sns.histplot(num_atoms_per_mol, bins=range(5, 31), ax=axes[0], color="steelblue")
axes[0].set_title("Atom Count per Molecule")
axes[0].set_xlabel("Number of atoms")

sns.histplot(homo_lumo_gap, bins=25, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("HOMO-LUMO Gap Distribution")
axes[1].set_xlabel("Gap (Hartree)")

# Bond order heatmap (channels 1..4 = SINGLE, DOUBLE, TRIPLE, AROMATIC) for molecule 0.
mol0_num_atoms = int(num_atoms_per_mol[0])
mol0_bonds = full_batch.bond_features[0, :mol0_num_atoms, :mol0_num_atoms]
bond_order_matrix = (1.0 - mol0_bonds[..., 0]) * (mol0_bonds[..., 1:].argmax(axis=-1) + 1)
sns.heatmap(bond_order_matrix, cmap="Blues", cbar=False, square=True, ax=axes[2])
axes[2].set_title(f"Bond Adjacency (mol 0, N={mol0_num_atoms})")
axes[2].set_xlabel("Atom index")
axes[2].set_ylabel("Atom index")

plt.tight_layout()
plt.show()